<a href="https://colab.research.google.com/github/DungeonProgger/IOS-Android/blob/main/Converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Инициализация и загрузка исходных данных
Исходный файл содержит сырые ответы респондентов в текстовом виде. В данном блоке данные загружаются в структуру DataFrame для их последующего преобразования в математическую модель, пригодную для машинного обучения.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='pandas')

import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

file_path = "Survey_Results_Plain.xlsx"
df = pd.read_excel(file_path)

print("Исходные данные загружены. Размер датасета:", df.shape)
df.head(3)

## 2. Исключение шума и кодирование целевой переменной
Целевая переменная (выбор между iOS и Android) переводится в бинарный числовой формат. Математические модели, включая K-ближайших соседей (KNN), не умеют работать со строковыми значениями, поэтому классы заменяются на 0 и 1.

In [ ]:
# Удаление признаков, не влияющих на предсказание
if 'Отметка времени' in df.columns:
    df.drop('Отметка времени', axis=1, inplace=True)

# Кодирование целевой переменной (Label Encoding)
target_col = '22. У Вас айфон или андроид?'
target_encoder = LabelEncoder()

df[target_col] = target_encoder.fit_transform(df[target_col].astype(str).str.strip().str.lower())

print("Распределение целевых классов (0 - Айфон, 1 - Андроид):")
print(df[target_col].value_counts())

## 3. Векторизация множественного выбора
В вопросе о факторах переплаты опрощенные могли выбирать несколько вариантов одновременно (ответы записаны через разделитель). Прямое кодирование таких ответов создало бы много уникальных, но ложных классов.

Чтобы модель могла оценивать вес каждого фактора в отдельности, применяется метод разделения: каждый возможный ответ (например, «Камера» или «Экосистема») выносится в независимый бинарный признак, где 1 означает выбор данного фактора, а 0 — его отсутствие.

In [ ]:
q4_col = '4. За что вы готовы переплатить, покупая смартфон?'

# Разделение строки по делителю и создание независимых бинарных признаков
q4_dummies = df[q4_col].astype(str).str.get_dummies(sep=';').add_prefix('Q4_')

# Объединение с основным датасетом и удаление исходной колонки
df = pd.concat([df, q4_dummies], axis=1)
df.drop(q4_col, axis=1, inplace=True)

print(f"Добавлено {q4_dummies.shape[1]} независимых признаков для вопроса 4.")

## 4. Обработка категорий и Z-масштабирование (StandardScaler)
Для подготовки данных к алгоритму KNN применяются два преобразования:

* **One-Hot Encoding (OHE)**: Оставшиеся текстовые данные (пол, сфера деятельности) конвертируются в нули и единицы. Если бы мы просто пронумеровали профессии (1, 2, 3), алгоритм считал бы одну профессию математически «больше» другой, что исказило бы расчет дистанций между пользователями. OHE устраняет эту ложную иерархию.
* **Масштабирование**: Алгоритм KNN базируется на вычислении евклидова расстояния. Без стандартизации признак с большими значениями (например, бюджет в 80 000) полностью подавит влияние признаков с малыми значениями (например, 2 зарядки в день). Z-масштабирование приводит все количественные переменные к единому распределению со средним 0 и дисперсией 1, уравнивая их вес при расчете расстояний.

In [ ]:
# Разделение признаков на числовые и категориальные
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
if target_col in numeric_cols:
    numeric_cols = numeric_cols.drop(target_col)

categorical_cols = df.select_dtypes(include=['object']).columns

# Устранение ложной иерархии (OHE)
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Уравнивание весов числовых признаков (Z-масштабирование)
scaler = StandardScaler()
if len(numeric_cols) > 0:
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# Приведение булевых типов к числовому формату
for col in df.columns:
    if df[col].dtype == bool:
        df[col] = df[col].astype(int)

print("Нормализация завершена. Итоговая размерность тензора:", df.shape)

In [ ]:
# Сохранение очищенного тензора данных
output_path = "NN_Training_Dataset.csv"
df.to_csv(output_path, index=False)

df.head(3)